# Machine Learning Preprocessing

This notebook prepares the cleaned NFHS-5 modeling dataset for machine learning.

It will:

- Load `df_model_v1.parquet`
- Remove identifiers and non-predictive survey fields
- Create a respondent-level train-test split
- Define numeric, categorical, and binary feature groups
- Handle missing values without leakage
- Encode categorical variables
- Create separate preprocessors for linear and tree-based models
- Verify and save the preprocessing objects

No final prediction model is trained in this notebook.

## 1. Import Libraries

In [8]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load the Processed Dataset

The cleaned dataset created in the data-preparation notebook is loaded from the saved Parquet file.

In [9]:
possible_paths = [
    Path("../data/processed/df_model_v1.parquet"),
    Path("data/processed/df_model_v1.parquet")
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "df_model_v1.parquet was not found. Run this notebook from the "
        "project root or the notebooks folder."
    )

df_model = pd.read_parquet(data_path)

print("Dataset shape:", df_model.shape)
display(df_model.head())

Dataset shape: (201311, 49)


,respondent_id,birth_index,cluster_number,household_number,respondent_line_number,sample_weight,primary_sampling_unit,sample_stratum_v022,sample_stratum_v023,state,...,urine_sample_during_pregnancy,blood_sample_during_pregnancy,told_about_pregnancy_complications,counselled_vaginal_bleeding,counselled_convulsions,counselled_prolonged_labour,counselled_severe_abdominal_pain,counselled_high_blood_pressure,convulsions_during_pregnancy,swelling_during_pregnancy
0,0100101399 02,1,113,99,2,193444,113,121,121,1,...,1,1,1,1,1,1,1,1,0,1
1,0100101357 02,1,113,57,2,193444,113,121,121,1,...,1,1,1,1,1,1,1,1,0,1
2,0100101358 04,1,113,58,4,193444,113,121,121,1,...,1,1,1,1,1,1,1,1,0,0
3,0100101380 02,1,113,80,2,193444,113,121,121,1,...,1,1,1,1,1,1,1,1,0,0
4,0100103022 04,1,130,22,4,201224,130,121,121,1,...,1,1,1,1,1,1,1,1,0,0


In [10]:
target_distribution = (
    df_model["csection"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Vaginal delivery",
        1: "Cesarean section"
    })
    .to_frame("count")
)

target_distribution["percent"] = (
    target_distribution["count"]
    / target_distribution["count"].sum()
    * 100
).round(2)

display(target_distribution)

print("Unique respondents:", df_model["respondent_id"].nunique())
print(
    "Rows belonging to respondents with multiple records:",
    df_model["respondent_id"].duplicated(keep=False).sum()
)

,count,percent
csection,,
Vaginal delivery,156569,77.77
Cesarean section,44742,22.23


Unique respondents: 158429
Rows belonging to respondents with multiple records: 82063


## 3. Separate Predictors and Target

Identifiers, survey-design variables, detailed geography, and variables that overlap with retained predictors are excluded.

`respondent_id` is kept separately only to prevent records from the same woman from appearing in both training and test sets.

In [11]:
candidate_excluded_columns = [
    "respondent_id",
    "birth_index",
    "cluster_number",
    "household_number",
    "respondent_line_number",
    "sample_weight",
    "primary_sampling_unit",
    "sample_stratum_v022",
    "sample_stratum_v023",
    "state",
    "district",
    "delivery_place_code",
    "survey_weight",
    "csection"
]

# Keep only columns that are present in the saved dataset
excluded_columns = [
    column
    for column in candidate_excluded_columns
    if column in df_model.columns
]

# Separate predictors, target, and respondent groups
X = df_model.drop(columns=excluded_columns).copy()

y = pd.to_numeric(
    df_model["csection"],
    errors="raise"
).astype(int)

groups = df_model["respondent_id"].copy()

print("Excluded columns:")
display(pd.Series(excluded_columns, name="Column"))

print("Predictor matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Number of predictor columns:", X.shape[1])

display(pd.Series(X.columns, name="Predictor"))

Excluded columns:


0              respondent_id
1                birth_index
2             cluster_number
3           household_number
4     respondent_line_number
5              sample_weight
6      primary_sampling_unit
7        sample_stratum_v022
8        sample_stratum_v023
9                      state
10                  district
11       delivery_place_code
12                  csection
Name: Column, dtype: str

Predictor matrix shape: (201311, 36)
Target vector shape: (201311,)
Number of predictor columns: 36


0                          facility_type
1                           maternal_age
2                        education_years
3                           wealth_index
4                              residence
5                               religion
6                           social_group
7                       health_insurance
8                            birth_order
9               total_children_ever_born
10              preceding_birth_interval
11                    age_at_first_birth
12                            twin_order
13                                   bmi
14                      first_anc_timing
15                            anc_visits
16                            anc_doctor
17                     anc_nurse_midwife
18             anc_traditional_attendant
19                  anc_community_worker
20                  anc_anganwadi_worker
21                       anc_asha_worker
22                    anc_other_provider
23                       no_anc_provider
24      weight_m

## 4. Define Feature Groups

Numeric variables will receive median imputation and missing indicators.

Nominal categorical variables will receive an explicit missing category and one-hot encoding.

Binary variables will retain `0`, `1`, and missing as separate categories so that missing values are not incorrectly treated as "No."

In [12]:
numeric_features = [
    "maternal_age",
    "education_years",
    "wealth_index",
    "birth_order",
    "total_children_ever_born",
    "preceding_birth_interval",
    "age_at_first_birth",
    "bmi",
    "first_anc_timing",
    "anc_visits"
]

categorical_features = [
    "facility_type",
    "residence",
    "religion",
    "social_group",
    "twin_order"
]

binary_features = [
    "health_insurance",
    "anc_doctor",
    "anc_nurse_midwife",
    "anc_traditional_attendant",
    "anc_community_worker",
    "anc_anganwadi_worker",
    "anc_asha_worker",
    "anc_other_provider",
    "no_anc_provider",
    "weight_measured_during_pregnancy",
    "bp_measured_during_pregnancy",
    "urine_sample_during_pregnancy",
    "blood_sample_during_pregnancy",
    "told_about_pregnancy_complications",
    "counselled_vaginal_bleeding",
    "counselled_convulsions",
    "counselled_prolonged_labour",
    "counselled_severe_abdominal_pain",
    "counselled_high_blood_pressure",
    "convulsions_during_pregnancy",
    "swelling_during_pregnancy"
]

defined_features = (
    numeric_features
    + categorical_features
    + binary_features
)

unassigned_features = sorted(
    set(X.columns) - set(defined_features)
)

duplicated_features = sorted({
    feature
    for feature in defined_features
    if defined_features.count(feature) > 1
})

missing_defined_features = sorted(
    set(defined_features) - set(X.columns)
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Binary features:", len(binary_features))
print("Unassigned predictors:", unassigned_features)
print("Duplicated assignments:", duplicated_features)
print("Defined features missing from X:", missing_defined_features)

if unassigned_features or duplicated_features or missing_defined_features:
    raise ValueError(
        "The feature-group definitions need to be corrected."
    )

Numeric features: 10
Categorical features: 5
Binary features: 21
Unassigned predictors: []
Duplicated assignments: []
Defined features missing from X: []


## 5. Convert Data to Scikit-learn-Compatible Types

The processed dataset contains pandas nullable data types such as `Int64`, `Float64`, and `pd.NA`. Some scikit-learn components do not handle these consistently.

The predictor matrix is rebuilt using standard NumPy and Python data types:

- Numeric variables are stored as ordinary `float64` values with `np.nan`.
- Categorical and binary variables are stored entirely as strings.
- Missing categorical and binary values are represented by the explicit category `"Missing"`.

This changes only the storage format and does not change the meaning of the variables.

In [13]:
# Rebuild the predictor matrix using standard NumPy and Python data types
def create_sklearn_compatible_frame(data):
    converted_data = pd.DataFrame(index=data.index)

    # Convert numeric variables to ordinary float64 values
    for feature in numeric_features:
        converted_data[feature] = np.asarray(
            pd.to_numeric(
                data[feature],
                errors="coerce"
            ),
            dtype=np.float64
        )

    # Convert categorical and binary variables to ordinary object strings
    for feature in categorical_features + binary_features:
        values = (
            data[feature]
            .astype("string")
            .fillna("Missing")
            .astype(str)
            .to_numpy(dtype=object)
        )

        converted_data[feature] = pd.Series(
            values,
            index=data.index,
            dtype=object
        )

    # Preserve the original predictor order
    return converted_data[data.columns]


# Convert the predictor matrix
X = create_sklearn_compatible_frame(X)

# Display the resulting data types
print("Predictor data types after conversion:")
display(X.dtypes.value_counts())


# Check for remaining pandas pd.NA values
columns_with_pd_na = []

for feature in X.columns:
    contains_pd_na = X[feature].map(
        lambda value: value is pd.NA
    ).any()

    if contains_pd_na:
        columns_with_pd_na.append(feature)

print("Columns containing pd.NA:", columns_with_pd_na)


# Check that categorical and binary columns contain only strings
mixed_type_columns = []

for feature in categorical_features + binary_features:
    observed_types = set(
        X[feature]
        .map(type)
        .unique()
    )

    if observed_types != {str}:
        mixed_type_columns.append({
            "feature": feature,
            "observed_types": observed_types
        })

print("Columns with mixed value types:")
display(pd.DataFrame(mixed_type_columns))


# Stop only when an actual incompatible value remains
if columns_with_pd_na or mixed_type_columns:
    raise TypeError(
        "Some predictors still contain incompatible values."
    )

print("All predictors are compatible with scikit-learn.")

Predictor data types after conversion:


object     26
float64    10
Name: count, dtype: int64

Columns containing pd.NA: []
Columns with mixed value types:


""


All predictors are compatible with scikit-learn.


## 6. Create a Respondent-Level Train-Test Split

The Birth Recode dataset may contain multiple records for the same woman. A normal row-level split could place records from one respondent in both sets.

`StratifiedGroupKFold` is therefore used to:

- Keep respondents separated
- Preserve the target distribution approximately
- Produce an approximately 80:20 split

In [14]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_indices, test_indices = next(
    splitter.split(
        X=X,
        y=y,
        groups=groups
    )
)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

groups_train = groups.iloc[train_indices].copy()
groups_test = groups.iloc[test_indices].copy()

print("Training predictors:", X_train.shape)
print("Test predictors:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training predictors: (161048, 36)
Test predictors: (40263, 36)
Training target: (161048,)
Test target: (40263,)


In [15]:
respondent_overlap = set(groups_train).intersection(
    set(groups_test)
)

print(
    "Respondents present in both training and test sets:",
    len(respondent_overlap)
)

if respondent_overlap:
    raise ValueError(
        "Respondent leakage detected between training and test sets."
    )

split_target_summary = pd.DataFrame({
    "training_percent": (
        y_train.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    ),
    "test_percent": (
        y_test.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )
}).round(2)

split_target_summary.index = [
    "Vaginal delivery",
    "Cesarean section"
]

display(split_target_summary)

Respondents present in both training and test sets: 0


,training_percent,test_percent
Vaginal delivery,77.77,77.77
Cesarean section,22.23,22.23


## 6. Compare Missingness Across the Split

No imputation is performed before the split. This check confirms that training and test missingness patterns are reasonably similar.

In [16]:
missingness_comparison = pd.DataFrame({
    "train_missing_percent": (
        X_train.isna().mean().mul(100)
    ),
    "test_missing_percent": (
        X_test.isna().mean().mul(100)
    )
})

missingness_comparison["absolute_difference"] = (
    missingness_comparison["train_missing_percent"]
    - missingness_comparison["test_missing_percent"]
).abs()

missingness_comparison = (
    missingness_comparison
    .sort_values(
        "train_missing_percent",
        ascending=False
    )
    .round(2)
)

display(missingness_comparison)

,train_missing_percent,test_missing_percent,absolute_difference
preceding_birth_interval,41.39,41.44,0.05
first_anc_timing,26.28,26.18,0.10
anc_visits,23.78,23.92,0.14
bmi,2.73,2.72,0.01
wealth_index,0.00,0.00,0.00
education_years,0.00,0.00,0.00
maternal_age,0.00,0.00,0.00
facility_type,0.00,0.00,0.00
health_insurance,0.00,0.00,0.00
residence,0.00,0.00,0.00


## 8. Build the Linear-Model Preprocessor

This preprocessor is designed for Logistic Regression and other models that benefit from standardized numeric features.

In [17]:
# Build the numeric pipeline for linear models
linear_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                missing_values=np.nan
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical values already contain the explicit "Missing" category
categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Binary values are represented as "0", "1", or "Missing"
binary_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Combine all transformations for linear models
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            linear_numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        )
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=True
)

linear_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and

## 9. Build the Tree-Model Preprocessor

Tree-based models do not require numeric scaling. The same imputation and encoding rules are used, but numeric features remain unscaled.

In [18]:
# Build the numeric pipeline for tree-based models
tree_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                missing_values=np.nan
            )
        )
    ]
)

# Combine all transformations for tree-based models
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        )
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=True
)

tree_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and

## 10. Fit and Transform the Data

Each preprocessor is fitted only on the training set. The test set is transformed using the rules learned from training data.

In [19]:
X_train_linear = linear_preprocessor.fit_transform(
    X_train
)

X_test_linear = linear_preprocessor.transform(
    X_test
)

X_train_tree = tree_preprocessor.fit_transform(
    X_train
)

X_test_tree = tree_preprocessor.transform(
    X_test
)

print("Linear training shape:", X_train_linear.shape)
print("Linear test shape:", X_test_linear.shape)
print("Tree training shape:", X_train_tree.shape)
print("Tree test shape:", X_test_tree.shape)

Linear training shape: (161048, 99)
Linear test shape: (40263, 99)
Tree training shape: (161048, 99)
Tree test shape: (40263, 99)


## 11. Verify the Transformed Data

The transformed datasets must have matching columns and contain no missing values.

In [20]:
def count_transformed_missing_values(matrix):
    if sparse.issparse(matrix):
        return np.isnan(matrix.data).sum()

    return np.isnan(matrix).sum()


assert X_train_linear.shape[0] == len(y_train)
assert X_test_linear.shape[0] == len(y_test)
assert X_train_tree.shape[0] == len(y_train)
assert X_test_tree.shape[0] == len(y_test)

assert X_train_linear.shape[1] == X_test_linear.shape[1]
assert X_train_tree.shape[1] == X_test_tree.shape[1]

verification_summary = pd.DataFrame({
    "dataset": [
        "X_train_linear",
        "X_test_linear",
        "X_train_tree",
        "X_test_tree"
    ],
    "rows": [
        X_train_linear.shape[0],
        X_test_linear.shape[0],
        X_train_tree.shape[0],
        X_test_tree.shape[0]
    ],
    "columns": [
        X_train_linear.shape[1],
        X_test_linear.shape[1],
        X_train_tree.shape[1],
        X_test_tree.shape[1]
    ],
    "missing_values": [
        count_transformed_missing_values(X_train_linear),
        count_transformed_missing_values(X_test_linear),
        count_transformed_missing_values(X_train_tree),
        count_transformed_missing_values(X_test_tree)
    ]
})

display(verification_summary)

,dataset,rows,columns,missing_values
0,X_train_linear,161048,99,0
1,X_test_linear,40263,99,0
2,X_train_tree,161048,99,0
3,X_test_tree,40263,99,0


In [21]:
linear_feature_names = (
    linear_preprocessor.get_feature_names_out()
)

tree_feature_names = (
    tree_preprocessor.get_feature_names_out()
)

print(
    "Number of linear-model features:",
    len(linear_feature_names)
)

print(
    "Number of tree-model features:",
    len(tree_feature_names)
)

print(
    "Feature names are identical:",
    np.array_equal(
        linear_feature_names,
        tree_feature_names
    )
)

display(
    pd.Series(
        linear_feature_names[:50],
        name="Transformed feature"
    )
)

Number of linear-model features: 99
Number of tree-model features: 99
Feature names are identical: True


0                                 numeric__maternal_age
1                              numeric__education_years
2                                 numeric__wealth_index
3                                  numeric__birth_order
4                     numeric__total_children_ever_born
5                     numeric__preceding_birth_interval
6                           numeric__age_at_first_birth
7                                          numeric__bmi
8                             numeric__first_anc_timing
9                                   numeric__anc_visits
10    numeric__missingindicator_preceding_birth_inte...
11                        numeric__missingindicator_bmi
12           numeric__missingindicator_first_anc_timing
13                 numeric__missingindicator_anc_visits
14                     categorical__facility_type_other
15                   categorical__facility_type_private
16                    categorical__facility_type_public
17                             categorical__resi

## 11. Record Correlated Feature Groups

The highly correlated variables identified during EDA are retained for the first baseline models. Reduced feature sets can later be compared against the full feature set.

No predictor is removed solely because of correlation at this stage.

In [22]:
correlated_feature_groups = {
    "reproductive_history": [
        "birth_order",
        "total_children_ever_born"
    ],
    "danger_sign_counselling": [
        "counselled_vaginal_bleeding",
        "counselled_convulsions",
        "counselled_prolonged_labour",
        "counselled_severe_abdominal_pain",
        "counselled_high_blood_pressure"
    ]
}

display(
    pd.DataFrame([
        {
            "group": group_name,
            "features": ", ".join(features)
        }
        for group_name, features
        in correlated_feature_groups.items()
    ])
)

,group,features
0,reproductive_history,"birth_order, total_children_ever_born"
1,danger_sign_counselling,"counselled_vaginal_bleeding, counselled_convul..."


## 13. Save the Preprocessing Objects

The fitted preprocessors, transformed feature names, and train-test split indices are saved for later modeling notebooks.

In [23]:
possible_artifact_paths = [
    Path("../artifacts"),
    Path("artifacts")
]

artifact_path = possible_artifact_paths[0]

if not artifact_path.parent.exists():
    artifact_path = possible_artifact_paths[1]

artifact_path.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    linear_preprocessor,
    artifact_path / "linear_preprocessor.joblib"
)

joblib.dump(
    tree_preprocessor,
    artifact_path / "tree_preprocessor.joblib"
)

pd.Series(
    linear_feature_names,
    name="feature_name"
).to_csv(
    artifact_path / "transformed_feature_names.csv",
    index=False
)

split_indices = pd.DataFrame({
    "row_index": df_model.index,
    "split": np.where(
        df_model.index.isin(X_train.index),
        "train",
        "test"
    )
})

split_indices.to_csv(
    artifact_path / "train_test_split_indices.csv",
    index=False
)

print("Preprocessing artifacts saved at:")
print(artifact_path.resolve())

Preprocessing artifacts saved at:
C:\IPD\IPD\artifacts


## 14. Preprocessing Summary

The preprocessing phase produced:

- A respondent-level training and test split
- No respondent overlap between the two sets
- Separate numeric, categorical, and binary feature groups
- Median imputation with missing indicators for numeric variables
- Explicit missing categories for categorical and binary variables
- One-hot encoding for categorical and binary features
- Standard scaling for linear-model numeric features
- An unscaled numeric pipeline for tree-based models
- Saved preprocessing objects and feature names

The dataset is now ready for baseline model development.